# Topic Modeling: French Legislative Election Manifestos

This notebook explores topic modeling on French legislative election manifestos using different methods.

## Setup and Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Topic modeling
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# Visualization
import pyLDAvis
import pyLDAvis.lda_model

# Set style
plt.style.use('default')
sns.set_palette("husl")

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
import spacy
! python -m spacy download fr_core_news_sm

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=7613) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 85.8 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


## Stopwords and lemmatization

In [4]:
STOPWORDS = [x.strip() for x in open('data/stop_word_fr.txt').readlines()]
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])

In [ ]:
# Eventuellement ajouter des stopwords spécifiques au corpus
# Add common French political terms that might not be informative
# additional_stopwords = {'loi', 'article', 'gouvernement', 'français', 'france', 
                        'monsieur', 'madame', 'nombre', 'partie', 'pays'}

## Load Metadata and Map Text Files to Candidates

In [5]:
# Load the CSV data - metadata
df = pd.read_csv('archelect_search.csv')

# Extract year and filter
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
df = df[df['contexte-election'] == "législatives"]

# Focus on 1973 for initial testing
year_of_interest = 1973
df_1973 = df[df['year'] == year_of_interest].copy()

print(f"Candidates in {year_of_interest}: {len(df_1973)}")
print(f"Columns available: {list(df.columns[:10])}...")
print(f"\nFirst few rows:")
df_1973.head()

Candidates in 1973: 3843
Columns available: ['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour', 'cote', 'departement', 'departement-nom', 'departement-insee']...

First few rows:


,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,year
10776,EL065_L_1973_03_001_01_1_PF_01,1973-03-04,Assemblée Nationale;France;Ve République;Élect...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,agriculteur,maire,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non,1973
10777,EL065_L_1973_03_001_01_1_PF_02,1973-03-04,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,entre 50 et 59 ans,non mentionné,conseiller général,non mentionné,non mentionné,non mentionné,Parti socialiste;Mouvement des radicaux de gauche,Union de la gauche socialiste et démocrate,non,1973
10778,EL065_L_1973_03_001_01_1_PF_03,1973-03-04,Assemblée Nationale;Ve République;Élections lé...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,entre 20 et 29 ans,extrudeur usine matières plastiques,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Faîtes confiance aux jeunes d'aujourd'hui,non,1973
10779,EL065_L_1973_03_001_01_1_PF_04,1973-03-04,France;Ve République;Élections législatives;As...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,non mentionné,instituteur honoraire,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,Union populaire et victoire du programme commun,oui,1973
10780,EL065_L_1973_03_001_01_1_PF_05,1973-03-04,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1973, Ain - 01, circ...",législatives,1,EL065,01,Ain,01 - Ain,...,entre 50 et 59 ans,négociant matériaux de construction,maire,non mentionné,sports et loisirs,résistant,non mentionné,Majorité Ve République,oui,1973


## Load Text Files for 1973

In [15]:
# Collect text files from 1973
text_dir = 'text_files/1973/legislatives'
text_files = []

if os.path.exists(text_dir):
    for file in os.listdir(text_dir):
        if file.endswith('.txt') and file[:-4] in df_1973['id'].values: # Ensure that we retrieve the doc in the metadata file
            text_files.append(os.path.join(text_dir, file))


print(f"Found {len(text_files)} text files from {year_of_interest}")

# Load all texts
documents = []
file_names = []

for file_path in text_files:
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            documents.append(content)
            file_names.append(os.path.basename(file_path))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"\nLoaded {len(documents)} documents")
print(f"Average document length: {np.mean([len(doc.split()) for doc in documents]):.0f} words")
print(f"Min length: {min([len(doc.split()) for doc in documents])} words")
print(f"Max length: {max([len(doc.split()) for doc in documents])} words")

Found 3843 text files from 1973

Loaded 3843 documents
Average document length: 807 words
Min length: 43 words
Max length: 2682 words


In [14]:
3921-3843

78

## Create Document-Term Matrix with CountVectorizer

In [8]:
n_features = 1000
STOPWORDS = [x.strip() for x in open('data/stop_word_fr.txt').readlines()]
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
lemmatized_texts = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(documents)]

tf_vectorizer = CountVectorizer(
    max_df=0.95, min_df=5, max_features=n_features, stop_words=STOPWORDS
)
doc_term_matrix = tf_vectorizer.fit_transform(lemmatized_texts)

## Latent Dirichlet Allocation

In [9]:
# Train LDA with different numbers of topics to find optimal
# Starting with 5 topics as a baseline
n_topics = 5
n_top_words = 15

print(f"Training LDA model with {n_topics} topics...")

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online',
    n_jobs=-1,
    verbose=0
)

lda_model.fit(doc_term_matrix)

print(f"Model trained! Perplexity: {lda_model.perplexity(doc_term_matrix):.4f}")

Training LDA model with 5 topics...
Model trained! Perplexity: 482.4628


### Display Top Words per Topic

In [11]:
def display_topics(model, feature_names, n_top_words=10):
    """
    Display the top words for each topic
    """
    print(f"\n{'='*80}")
    print(f"TOP {n_top_words} WORDS PER TOPIC (LDA with {model.n_components} topics)")
    print(f"{'='*80}\n")
    
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_words_idx]
        weights = topic[top_words_idx]
        
        print(f"Topic {topic_idx + 1}:")
        for word, weight in zip(top_words, weights):
            print(f"  {word:20s} {weight:.4f}")
        print()

vocab = tf_vectorizer.get_feature_names_out()
display_topics(lda_model, vocab, n_top_words)


TOP 15 WORDS PER TOPIC (LDA with 5 topics)

Topic 1:
  social               4891.4516
  pouvoir              4646.6688
  devoir               3618.4662
  politique            3582.9279
  france               3167.3217
  vie                  3010.3160
  national             2664.1535
  français             2523.2613
  homme                2502.0642
  économique           2231.4000
  grand                2181.4711
  droit                2144.3252
  circonscription      2120.3001
  travail              2087.0576
  falloir              2080.7420

Topic 2:
  pouvoir              2460.2905
  travailleur          1646.6014
  socialisme           1569.5843
  autogestion          1393.7010
  société              1334.6589
  français             1181.0628
  travail              1080.1848
  vie                  1042.3742
  populaire            1040.7833
  psu                  1028.8917
  socialiste           1009.6988
  lutte                967.8887
  ouvrier              959.1750
  actuel      

### Interactive Visualization with pyLDAvis

In [ ]:
pyLDAvis.enable_notebook()
pyLDAvis.lda_model.prepare(lda_model, doc_term_matrix, tf_vectorizer)

## Non-negative matrix factorization

In [ ]:
# Encode the text with TF-IDF vecotrize
tfidf_vectorizer = TfidfVectorizer(max_features=n_features, stop_words=STOPWORDS)
tfidf = tfidf_vectorizer.fit_transform(lemmatized_texts)

# Define the NMF model
nmf = NMF(n_topics)

# Train the NMF
W = nmf.fit_transform(tfidf)

# print the words for each topic
feature_names = tfidf_vectorizer.get_feature_names_out()
n_top_words = 20

for topic_idx, topic in enumerate(nmf.components_):
    top_indices = np.argsort(-topic)[:n_top_words]
    top_words = [feature_names[i] for i in top_indices]
    
    print(f"Topic {topic_idx}:")
    print("  " + ", ".join(top_words))
    print()

# topic_label = {}
# topic_label[0] = 'Théâtre'
# topic_label[1] = 'Horaire'
# topic_label[2] = 'Cinéma'
# topic_label[3] = 'Musique pop/rock'
# topic_label[4] = 'Musée'
# topic_label[5] = 'Syndicalisme'
# topic_label[6] = 'USA'
# topic_label[7] = 'Musique classique'
# topic_label[8] = 'Culture'
# topic_label[9] = 'Marché'


# Possible to do document/topic analysis -> see the distribution of topic for a given doc
# --> Could extend it to see the distribution of topic for a given party 
# --> See the evaluation of this distribution throughout years

In [ ]:
# Normalize the W matrix
W_normalized = W/np.sum(W, axis=1, keepdims=True)
df["dominant_topic"] = np.argmax(W_normalized, axis=1)
df["topic_score"] = np.max(W_normalized, axis=1)

In [ ]:
import textwrap

topic_names = [f"{k}-{topic_label[k]}" for k in topic_label.keys()]
    
for row in df.sample(n=5, random_state=2026).itertuples():
    
    doc_id = row.Index
    topic_distribution = W_normalized[doc_id]
    dominant_topic = row.dominant_topic
    dominant_score = row.topic_score
    dominant_label = topic_label[dominant_topic]
    
    print(f"\nDocument {doc_id}")
    print(f"Dominant topic: {dominant_topic} - {dominant_label} ({dominant_score:.3f})\n")
    text = textwrap.fill(row.text, width=100)
    print(text[:1200])
    print("\n\n")
    
    # Topic distribution
    plt.figure(figsize=(12,4))
    bars = plt.bar(topic_names, topic_distribution)
    bars[dominant_topic].set_color("darkred")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Proportion")
    plt.title(f"Topic mixture for document {doc_id}")
    plt.tight_layout()
    plt.show()

## Bertopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

docs = df["text"].tolist()

# Encode the documents with SentenceTransformer
sentence_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=True)

# Create a CountVectorizer with French stop words
vectorizer = CountVectorizer(stop_words=STOPWORDS, ngram_range=(1, 2))

# Create the BERTopic model using the vectorizer and fit it
topic_model = BERTopic(embedding_model=sentence_model, vectorizer_model=vectorizer, nr_topics=n_topics)
topics, probs = topic_model.fit_transform(df["text"], embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_documents(
    docs=docs,
    embeddings=embeddings,
    hide_annotations=True,
    topics=[0, 1, 2, 3, 4, 5,6, 7, 8, 9],
    height=600,
    width=1000
)

## Selecting the good number of topics

In [ ]:
from gensim.corpora import Dictionary

analyzer = tfidf_vectorizer.build_analyzer()

tokenized_texts = [
    analyzer(doc)
    for doc in df.lemmatized_text
]


dictionary = Dictionary(tokenized_texts)

# Optional but recommended:
dictionary.filter_extremes(no_below=5, no_above=0.95)


In [ ]:
from gensim.models.coherencemodel import CoherenceModel

def get_topics(model, feature_names, n_top_words):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        # Get indices of top words (descending order)
        top_indices = np.argsort(-topic)[:n_top_words]
        # Map indices to words
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics

feature_names = tfidf_vectorizer.get_feature_names_out()

# for each number of topics, we will compute the 4 metrics
topic_range = [5, 10, 15, 20, 30]
coherence_metrics = ['u_mass', 'c_v', 'c_uci', 'c_npmi']
results={}
for metrics in coherence_metrics:
    results[metrics] = {}


for k in topic_range:
    # fit the model with k topics
    nmf = NMF(
        n_components=k,
        random_state=42
    )

    W = nmf.fit_transform(tfidf)
    topics = get_topics(nmf, feature_names, 10)
    for met in coherence_metrics:

        coherence_model = CoherenceModel(
            topics=topics,
            texts=tokenized_texts,
            dictionary=dictionary,
            coherence=met
        )

        coherence_score = coherence_model.get_coherence()
        results[met][k] =  coherence_score
        print(f"NMF coherence for {k} topic and metrics {met}: {coherence_score}")

In [ ]:

plt.figure(figsize=(10, 6))

for met in coherence_metrics:
    ks = sorted(results[met].keys())
    scores = np.array([results[met][k] for k in ks])

    # Min-max normalize per metric
    norm_scores = (scores - scores.min()) / (scores.max() - scores.min())

    plt.plot(ks, norm_scores, marker='o', label=met)

plt.xlabel("Number of Topics (k)")
plt.ylabel("Normalized Coherence Score")
plt.title("Normalized Coherence Comparison")
plt.legend()
plt.grid(True)
plt.show()